# Test-content / infrastructure issue detector
Flags failures that are *not* product bugs, only at ≥90 % precision (a false positive would hide a real bug). Two arms: stored, precision-pruned rules (shown by name in the UI) and a TF-IDF + categorical logistic regression for what the rules miss.

> All data in this notebook is **synthetic** (generated below). The real system runs on private validation data; the code paths are the same, the numbers in the README are from the real data.

In [ ]:
!pip -q install scikit-learn matplotlib numpy scipy

In [ ]:
%%writefile synthetic.py
"""Synthetic failure generator for the demo notebooks (no real data is published).
Each bug has a signature (2-3 log lines), a domain, an owner and a test family; runs of that bug emit the signature
mixed with background noise, sometimes miss a line, and 'sibling' bugs share most of their signature, which is the
hard case in the real system. Not-a-bug runs are infrastructure or test-content problems."""
import random
from datetime import datetime, timedelta

DOMAINS = ["power", "memory", "reset", "virtualization", "media", "compute", "fabric", "firmware"]
COMPONENTS = ["gt", "dma", "pcie", "fw", "mem", "sched", "link", "irq", "pmu", "vf"]
SYMPTOMS = ["engine reset", "timeout waiting for fence", "page fault at <HEX>", "device lost", "link training failed",
            "ECC uncorrectable", "watchdog expired", "CRC mismatch", "thermal trip", "context banned"]
NOISE = ["test step <N> started", "allocating <N> buffers", "INFO: iteration <N> passed", "loading module", "heartbeat ok",
         "perf counter sample <N>", "config applied", "mounting share", "INFO: retrying connection"]
GENERIC = ["FAIL: test step <N> returned non-zero", "ERROR: timeout waiting for test harness", "FAIL: assertion in test script"]
INFRA = ["ERROR: node unreachable", "ERROR: container image pull failed", "FAIL: missing test binary", "ERROR: disk quota exceeded"]


def make_world(n_bugs=40, n_runs=1200, p_none=.18, seed=0):
    rng = random.Random(seed)
    t0 = datetime(2026, 1, 1)
    bugs = []
    for i in range(n_bugs):
        sib = bugs[rng.randrange(len(bugs))] if bugs and rng.random() < .35 else None
        comp = sib["component"] if sib else rng.choice(COMPONENTS)
        sig = list(sib["signature"][:2]) if sib else [f"{comp}: {rng.choice(SYMPTOMS)}", f"{comp}: {rng.choice(SYMPTOMS)} on <PCI>"]
        sig.append(f"{comp}: first fault code {rng.randrange(16, 255):#x}")
        bugs.append({"id": f"BUG-{i:03d}", "domain": sib["domain"] if sib else rng.choice(DOMAINS), "component": comp,
                     "owner": f"owner{rng.randrange(8)}", "family": f"T{rng.randrange(12)}", "signature": sig,
                     "created": t0 + timedelta(days=rng.randrange(0, 200)),
                     "title": f"[{comp}] {sig[0].split(': ', 1)[1]}", "root_cause": f"{comp} {sig[-1].split(': ', 1)[1]}"})
    w = [1 / (i + 1) ** .8 for i in range(n_bugs)]          # long tail: a few bugs own most failures
    runs = []
    for j in range(n_runs):
        start = t0 + timedelta(days=rng.uniform(0, 260))
        if rng.random() < p_none:
            lines, label, b = ([rng.choice(INFRA)] if rng.random() < .6 else [rng.choice(GENERIC)]), "NONE", None
        else:
            b = rng.choices(bugs, w)[0]
            if b["created"] > start + timedelta(days=30):
                start = b["created"] + timedelta(days=rng.uniform(0, 30))
            lines, label = [s for s in b["signature"] if rng.random() > .3] or b["signature"][:1], b["id"]
            if rng.random() < .25:
                lines.append(rng.choice(GENERIC + INFRA))      # product bugs also trip generic/infra-looking lines
        lines += rng.sample(NOISE, 4)
        rng.shuffle(lines)
        runs.append({"run": f"RUN-{j:05d}", "label": label, "start": start, "lines": lines,
                     "domain": b["domain"] if b else rng.choice(DOMAINS), "owner": b["owner"] if b else f"owner{rng.randrange(8)}",
                     "family": b["family"] if b and rng.random() < .7 else f"T{rng.randrange(12)}"})
    return bugs, runs


def split(runs, bugs, seed=0):
    """Grouped by bug: 30% of bugs are test-only, so the test split measures bugs never seen in training."""
    rng = random.Random(seed)
    ids = [b["id"] for b in bugs]
    rng.shuffle(ids)
    test_bugs = set(ids[: int(.3 * len(ids))])
    for r in runs:
        r["split"] = "test" if (r["label"] in test_bugs or (r["label"] == "NONE" and rng.random() < .3)) else "train"
        # frequent bugs also appear in test (seen-bug slice)
        if r["label"] not in test_bugs and r["label"] != "NONE" and rng.random() < .2:
            r["split"] = "test"
    return runs

In [ ]:
%%writefile detector.py
"""Test-content / infrastructure issue detector: is this failure *not* a product bug?

A false positive hides a real bug, so the model only fires at an operating point with >= 90 % out-of-fold
precision. Features: TF-IDF over the failure reason, the failed stage and the error lines of the execution log
(numbers and paths masked), plus one-hot categorical flags (test family, owner, domain, evidence present or not).
Labels: runs humans tagged as test-content issues (positives) vs runs linked to a product bug (negatives).

Two arms behind one verdict (Triage): editable, stored rules (precision-pruned, each tagged infra or test_content, shown by name in the
UI) run first; the logistic regression above is the model arm for what the rules do not cover.
"""
from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import OneHotEncoder

CATS = ["family", "owner", "domain", "has_logs"]


class IssueDetector:
    def __init__(self, C: float = 3.0, min_precision: float = .9):
        self.C, self.min_precision = C, min_precision

    def _text(self, rows):
        return [" ".join(r["lines"]) for r in rows]

    def _cats(self, rows):
        return [[str(r.get(c, "")) for c in CATS] for r in rows]

    def _x(self, rows, fit=False):
        if fit:
            self.tf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
            self.oh = OneHotEncoder(handle_unknown="ignore")
            return hstack([self.tf.fit_transform(self._text(rows)), self.oh.fit_transform(self._cats(rows))]).tocsr()
        return hstack([self.tf.transform(self._text(rows)), self.oh.transform(self._cats(rows))]).tocsr()

    def fit(self, rows, y, groups):
        """Grouped CV gives out-of-fold scores; the threshold is the best-recall point with OOF precision >= target."""
        oof = np.zeros(len(rows))
        for tr, te in GroupKFold(5).split(rows, y, groups):
            m = LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced")
            m.fit(self._x([rows[i] for i in tr], fit=True), np.array(y)[tr])
            oof[te] = m.predict_proba(self._x([rows[i] for i in te]))[:, 1]
        p, r, t = precision_recall_curve(y, oof)
        ok = [(r[i], t[i]) for i in range(len(t)) if p[i] >= self.min_precision]
        self.tau = max(ok)[1] if ok else 1.0
        self.oof_ap = average_precision_score(y, oof)
        self.oof_recall = max(ok)[0] if ok else 0.0
        self.model = LogisticRegression(C=self.C, max_iter=3000, class_weight="balanced").fit(self._x(rows, fit=True), y)
        return self

    def flag(self, row) -> tuple[bool, float]:
        s = float(self.model.predict_proba(self._x([row]))[:, 1][0])
        return s >= self.tau, s


@dataclass
class Rule:
    """One stored triage rule: fires when any `any_of` phrase is in the failure reason / log and no `none_of` phrase is."""
    name: str
    kind: str                                   # "infra" | "test_content"
    any_of: list[str]
    none_of: list[str] = field(default_factory=list)

    def fires(self, row) -> bool:
        t = " ".join(row["lines"]).lower()
        return any(p in t for p in self.any_of) and not any(p in t for p in self.none_of)


class RuleTriage:
    """Rules live in a shared store (users can add their own); base rules are pruned by measured precision."""

    def __init__(self, rules: list[Rule]):
        self.rules = rules

    def prune(self, rows, y, min_precision: float = .9):
        """Keep a rule only if its precision on labelled runs (non-bug = positive) is at least `min_precision`."""
        keep = []
        for ru in self.rules:
            hit = [yy for r, yy in zip(rows, y) if ru.fires(r)]
            if hit and sum(hit) / len(hit) >= min_precision:
                keep.append(ru)
        self.rules = keep
        return self

    def match(self, row):
        return next((ru for ru in self.rules if ru.fires(row)), None)


class Triage:
    """Rules first (explainable by rule name), then the model arm at its >= 90% precision threshold."""

    def __init__(self, rules: RuleTriage, model: IssueDetector):
        self.rules, self.model = rules, model

    def flag(self, row) -> tuple[bool, str]:
        ru = self.rules.match(row)
        if ru:
            return True, f"rule:{ru.name} ({ru.kind})"
        hit, score = self.model.flag(row)
        return hit, f"model:{score:.2f}"

In [ ]:
import random
from synthetic import make_world
from detector import IssueDetector, Rule, RuleTriage, Triage
bugs, runs = make_world(n_bugs=40, n_runs=2000, p_none=.25, seed=2)
rng = random.Random(0)
for r in runs:
    r["has_logs"] = rng.random() > .1
    if r["label"] != "NONE" and rng.random() < .08:      # some product bugs also log infrastructure-looking lines
        r["lines"].append("ERROR: node unreachable")
y = [int(r["label"] == "NONE") for r in runs]
groups = [r["label"] if r["label"] != "NONE" else r["family"] for r in runs]
det = IssueDetector().fit(runs, y, groups)
print(f"OOF average precision {det.oof_ap:.2f} | recall at P>=0.9: {det.oof_recall:.2f} | threshold {det.tau:.2f}")

## Stored rules, pruned by measured precision
A candidate rule survives only if its precision on labelled runs is at least 90%. Rules carry a not-contains guard against product signatures. The generic timeout rule below has no guard and also matches real bugs, so pruning drops it.

In [ ]:
PRODUCT = ["engine reset", "page fault", "device lost", "link training", "ecc", "watchdog", "crc mismatch", "thermal", "banned", "fence", "first fault"]
candidates = [
    Rule("node unreachable", "infra", ["node unreachable"], PRODUCT),            # "and not contains a product signature"
    Rule("image not pullable", "infra", ["image pull failed"], PRODUCT),
    Rule("missing test binary", "infra", ["missing test binary"], PRODUCT),
    Rule("test script assertion", "test_content", ["assertion in test script"], PRODUCT),
    Rule("any timeout", "infra", ["timeout waiting for"]),                      # sounds right, also matches real bugs
]
rt = RuleTriage(candidates).prune(runs, y)
print("kept:", [ru.name for ru in rt.rules])
tri = Triage(rt, det)
flags = [(tri.flag(r), r["label"] == "NONE") for r in runs]
caught = [(why, pos) for (hit, why), pos in flags if hit]
print(f"flagged {len(caught)} | precision {sum(p for _, p in caught) / len(caught):.1%} | by rule: "
      f"{sum(w.startswith('rule') for w, _ in caught)}, by model: {sum(w.startswith('model') for w, _ in caught)}")